# Moderation & Interaction Analysis

调节效应分析：人格特质 × AI调用行为的交互作用，及 cluster 水平的简单斜率检验。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

In [ ]:
# 输入文件路径
output_dir = Path('output')
trial_with_cluster_file = output_dir / 'trial_with_cluster.csv'

data_dir = Path('../../../data/analysis')
performance_file = data_dir / 'performance' / 'performance.csv'

In [ ]:
# 工具函数

# ---- GLMM 两两比较（基于固定效应估计，在对数链接尺度上）----
# 说明：针对前面拟合的 PoissonBayesMixedGLM（fluency 与 above_median），
# 计算不同 cluster 水平之间的两两对比：估计差值（link），标准误，z / p，
# 以及对数链接尺度的 95% 置信区间与 IRR（exp(diff)）区间。
import patsy
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import numpy as np
from scipy import stats as _sps
import pandas as _pd

def pairwise_glmm_contrasts(glmm, glmm_res, df, formula='C(cluster) + dat_score + ribs_total + cse_total + ai_attitude', cluster_col='cluster', covariates=None, model_label='model'):
    exog_names = list(glmm.exog_names)
    k_fe = len(exog_names)

    # 获取固定效应及其协方差：VB 有 fe_mean / fe_sd，MAP 有 params / cov_params
    if hasattr(glmm_res, 'fe_mean'):
        fe_mean = np.asarray(glmm_res.fe_mean).reshape(-1)
        fe_sd = np.asarray(glmm_res.fe_sd).reshape(-1)
        cov_fe = np.diag(fe_sd ** 2)
        cov_fe = _pd.DataFrame(cov_fe, index=exog_names, columns=exog_names)
    else:
        all_params = np.asarray(glmm_res.params).reshape(-1)
        fe_mean = all_params[:k_fe]
        try:
            cov = np.asarray(glmm_res.cov_params())
            cov_fe = _pd.DataFrame(cov, index=glmm.exog_names, columns=glmm.exog_names)
            cov_fe = cov_fe.reindex(index=exog_names, columns=exog_names).fillna(0.0)
        except Exception:
            cov_fe = _pd.DataFrame(np.nan, index=exog_names, columns=exog_names)

    # cluster 水平
    if isinstance(df[cluster_col].dtype, pd.CategoricalDtype):
        clusters = df[cluster_col].cat.categories
    else:
        clusters = sorted(df[cluster_col].dropna().unique())

    pred_df = _pd.DataFrame({cluster_col: clusters})
    # 填充协变量（默认使用样本均值），covariates 可以传入 dict 覆盖
    if covariates is None:
        covariates = {}
    for cov, val in covariates.items():
        if val is None and cov in df.columns:
            pred_df[cov] = df[cov].mean()
        else:
            pred_df[cov] = val

    # 若连续协变量未传且存在于 df 中，则设为均值
    for cov in ['dat_score', 'ribs_total', 'cse_total', 'ai_attitude']:
        if cov not in pred_df.columns and cov in df.columns:
            pred_df[cov] = df[cov].mean()

    design = patsy.dmatrix(formula, pred_df, return_type='dataframe')
    # 保证 design 列与 exog_names 对齐
    for col in exog_names:
        if col not in design.columns:
            design[col] = 0.0
    design = design[exog_names]

    results = []
    for i, j in combinations(range(len(clusters)), 2):
        xa = design.iloc[i].values
        xb = design.iloc[j].values
        contrast = xa - xb
        diff = float(contrast.dot(fe_mean))
        # 计算方差（若协方差包含 NaN 则返回 NaN）
        try:
            var = float(contrast.dot(cov_fe.values).dot(contrast))
        except Exception:
            var = np.nan
        se = np.sqrt(var) if not np.isnan(var) else np.nan
        zstat = diff / se if (se and not np.isnan(se)) else np.nan
        pval = 2 * _sps.norm.sf(abs(zstat)) if not np.isnan(zstat) else np.nan
        # 95% CI 在 link（对数）尺度上，再转换到 IRR（exp）尺度
        if not np.isnan(se):
            ci_lo = diff - 1.96 * se
            ci_hi = diff + 1.96 * se
            irr = np.exp(diff)
            irr_lo = np.exp(ci_lo)
            irr_hi = np.exp(ci_hi)
        else:
            ci_lo = ci_hi = irr = irr_lo = irr_hi = np.nan
        results.append({
            'group1': clusters[i],
            'group2': clusters[j],
            'est_link': diff,
            'se': se,
            'z': zstat,
            'p': pval,
            'ci_lo_link': ci_lo,
            'ci_hi_link': ci_hi,
            'irr': irr,
            'irr_lo': irr_lo,
            'irr_hi': irr_hi
        })

    res_df = _pd.DataFrame(results).sort_values('p')
    if len(res_df):
        res_df['p_bonf'] = multipletests(res_df['p'].values, method='bonferroni')[1]
        res_df['p_fdr'] = multipletests(res_df['p'].values, method='fdr_bh')[1]
    print(f'--- {model_label} 两两比较（按 p 排序）---')
    display(res_df.round(4))
    return res_df


In [ ]:
# 加载数据
df_cluster = pd.read_csv(trial_with_cluster_file)  # 包含 participant_id, trial_index, cluster 与行为特征
df_perf = pd.read_csv(performance_file)             # 包含 participant_id, item_name, trial_index, fluency, originality

# 以试次为单位合并聚类标签
df_merged = pd.merge(
    df_perf,
    df_cluster,
    on=['participant_id', 'trial_index'],
    how='left'
 )

print(df_merged['cluster'].value_counts(dropna=False).sort_index())
df_merged.describe()

In [ ]:
# 中文标签映射（供调节分析使用）
corr_labels_cn = {
    'trial_calls': 'AI调用次数',
    'first_ai_time': '首次调用时间',
    'pre_first_call_ideas': '首次调用前想法数',
    'pre_think_time': '平均调用前思考时间',
    'perspective_taking': '观点采纳率',
    'affected_by_ai': '受AI影响率',
    'bfi_extroversion': '外向性',
    'bfi_agreeableness': '宜人性',
    'bfi_conscientiousness': '尽责性',
    'bfi_neuroticism': '神经质',
    'bfi_openness': '开放性',
    'cse_total': '创造性自我效能',
    'ribs_total': 'RIBS总分',
    'ai_attitude': 'AI态度',
    'dat_score': 'DAT分数',
    'originality': '原创性',
    'fluency': '流畅性',
    'above_median': '高质量回答数',
    'above_median_ratio': '高质量回答率',
    'cluster': '簇别'
}


### 调节分析

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from IPython.display import display as ipy_display

def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)

mod_df = df_merged[df_merged["cluster"] >= 0].copy()
for col in ["trial_calls", "first_ai_time", "cse_total", "ribs_total", "dat_score", "bfi_openness", "ai_attitude", "originality", "fluency"]:
    if col in mod_df.columns:
        mod_df[f"{col}_z"] = zscore(mod_df[col])

def _fit_mixed_or_ols(formula, data, has_groups, model_label=""):
    """Try MixedLM first, fall back to OLS if convergence fails."""
    if has_groups:
        try:
            model = sm.MixedLM.from_formula(
                formula,
                groups="participant_id",
                vc_formula={"item_name": "0 + C(item_name)"},
                data=data
            )
            result = model.fit(reml=False, method="lbfgs", maxiter=2000)
            return result, "LMM"
        except Exception as e:
            print(f"  LMM failed ({e}), falling back to OLS")
    model = ols(formula, data=data).fit()
    return model, "OLS"


def _fit_and_jn(predictor, moderator, outcome, data, controls=["age", "gender"], alpha=0.05, verbose=False):
    """拟合交互模型，返回 JN 简单斜率数据。"""
    import patsy as _patsy

    p_z = f"{predictor}_z" if f"{predictor}_z" in data.columns else predictor
    m_z = f"{moderator}_z" if f"{moderator}_z" in data.columns else moderator

    subset = data.dropna(subset=[p_z, m_z, outcome] + controls).copy()
    if len(subset) < 30:
        return None

    ctrl_str = " + ".join(controls)
    rhs = f"{p_z} * {m_z} + {ctrl_str}"
    formula = f"{outcome} ~ {rhs}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    model, mtype = _fit_mixed_or_ols(formula, subset, has_groups)

    # ---- 提取系数与协方差 ----
    if mtype == "LMM":
        params = model.fe_params
        cov_raw = model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov_fe = cov_raw.loc[params.index, params.index]
        else:
            cov_fe = pd.DataFrame(cov_raw, index=model.model.exog_names, columns=model.model.exog_names)
            cov_fe = cov_fe.loc[params.index, params.index]

        main_formula = f"{outcome} ~ {p_z} + {m_z} + {ctrl_str}"
        try:
            model_main, _ = _fit_mixed_or_ols(main_formula, subset, has_groups)
            lr_stat = 2 * (model.llf - model_main.llf)
            p_inter = stats.chi2.sf(lr_stat, 1)
        except Exception:
            p_inter = np.nan
    else:
        params = model.params
        cov_raw = model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov_fe = cov_raw.loc[params.index, params.index]
        else:
            cov_fe = pd.DataFrame(cov_raw, index=params.index, columns=params.index)

        main_formula = f"{outcome} ~ {p_z} + {m_z} + {ctrl_str}"
        model_main = ols(main_formula, data=subset).fit()
        anova_res = sm.stats.anova_lm(model_main, model)
        p_inter = anova_res["Pr(>F)"].iloc[1]

    # ---- 找到交互项 ----
    inter_terms = [k for k in params.index if ":" in k and p_z in k]
    if not inter_terms:
        inter_terms = [k for k in params.index if ":" in k]
    if not inter_terms:
        if verbose: print(f"  [JN] 未找到交互项，params={list(params.index)}")
        return None
    inter_key = inter_terms[0]

    b_pred = params.get(p_z)
    b_inter = params.get(inter_key)
    if b_pred is None or b_inter is None:
        if verbose: print(f"  [JN] 找不到系数: {p_z}={b_pred}, {inter_key}={b_inter}")
        return None

    try:
        v_pred = cov_fe.at[p_z, p_z]
        v_inter = cov_fe.at[inter_key, inter_key]
        cov_pi = cov_fe.at[p_z, inter_key]
    except KeyError as e:
        if verbose: print(f"  [JN] 协方差查找失败: {e}")
        return None

    if verbose:
        print(f"  [{predictor} x {moderator} -> {outcome}] {mtype}")
        print(f"    b_pred({p_z}) = {b_pred:.4f}, v = {v_pred:.6f}")
        print(f"    b_inter({inter_key}) = {b_inter:.4f}, v = {v_inter:.6f}")
        print(f"    cov(pred, inter) = {cov_pi:.6f}")

    # ---- JN 计算 ----
    z_crit = stats.norm.ppf(1 - alpha / 2)

    m_raw = subset[moderator]
    m_model = subset[m_z]
    m_raw_min, m_raw_max = m_raw.min(), m_raw.max()
    m_model_min, m_model_max = m_model.min(), m_model.max()

    m_range = np.linspace(m_raw_min, m_raw_max, 300)
    m_model_range = np.linspace(m_model_min, m_model_max, 300)

    simple_slope = b_pred + b_inter * m_model_range
    var_slope = v_pred + m_model_range**2 * v_inter + 2 * m_model_range * cov_pi
    se_slope = np.sqrt(np.maximum(var_slope, 0))
    ci_lo = simple_slope - z_crit * se_slope
    ci_hi = simple_slope + z_crit * se_slope
    sig_mask = (ci_lo > 0) | (ci_hi < 0)

    # ---- 解 JN 边界点 ----
    A = b_inter**2 - z_crit**2 * v_inter
    B = 2 * (b_pred * b_inter - z_crit**2 * cov_pi)
    C = b_pred**2 - z_crit**2 * v_pred

    jn_points_raw = []
    if abs(A) > 1e-12:
        disc = B**2 - 4 * A * C
        if disc >= 0:
            sqrt_disc = np.sqrt(disc)
            for root_model in [(-B + sqrt_disc) / (2 * A), (-B - sqrt_disc) / (2 * A)]:
                if m_model_min <= root_model <= m_model_max:
                    root_raw = np.interp(root_model, [m_model_min, m_model_max], [m_raw_min, m_raw_max])
                    jn_points_raw.append(root_raw)

    if verbose:
        print(f"    斜率范围: [{simple_slope.min():.3f}, {simple_slope.max():.3f}]")
        print(f"    显著比例: {sig_mask.mean():.1%}")
        print(f"    JN 边界: {jn_points_raw}")

    return {
        "m_range": m_range, "simple_slope": simple_slope,
        "ci_lo": ci_lo, "ci_hi": ci_hi, "sig_mask": sig_mask,
        "jn_points": sorted(jn_points_raw), "p_inter": p_inter,
        "p_cn": corr_labels_cn.get(predictor, predictor),
        "m_cn": corr_labels_cn.get(moderator, moderator),
        "y_cn": corr_labels_cn.get(outcome, outcome),
        "mtype": mtype, "m_raw": subset[moderator],
        "p_raw": subset[p_z], "y_raw": subset[outcome],
        "m_mean": m_raw.mean(), "m_sd": m_raw.std(ddof=0)
    }


def interaction_analysis(predictor, moderator, outcome, controls=["age","gender"], do_jn=False, plot_cat_slopes=False):
    needed = [predictor, moderator, outcome] + controls
    missing = [c for c in needed if c not in mod_df.columns and f"{c}_z" not in mod_df.columns]
    if missing:
        print(f"跳过 {predictor}x{moderator}->{outcome}：缺少字段: {missing}")
        return

    p_z = f"{predictor}_z" if f"{predictor}_z" in mod_df.columns else predictor
    m_z = f"{moderator}_z" if f"{moderator}_z" in mod_df.columns else moderator
    y = outcome

    subset = mod_df.dropna(subset=[p_z, m_z, y] + controls)
    if len(subset) < 30:
        print(f"样本量过小 ({len(subset)})，跳过 {predictor}x{moderator}->{outcome}")
        return

    if moderator == "cluster":
        subset[moderator] = subset[moderator].astype("category")

    ctrl_list = [c for c in controls if c in subset.columns]
    ctrl_str = " + ".join(ctrl_list) if ctrl_list else ""

    if ctrl_str:
        main_formula = f"{y} ~ {p_z} + {m_z} + {ctrl_str}"
        int_formula = f"{y} ~ {p_z} * {m_z} + {ctrl_str}"
    else:
        main_formula = f"{y} ~ {p_z} + {m_z}"
        int_formula = f"{y} ~ {p_z} * {m_z}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns

    model_main, mtype = _fit_mixed_or_ols(main_formula, subset, has_groups)
    model_int, _ = _fit_mixed_or_ols(int_formula, subset, has_groups)

    p_inter = None
    if mtype == "LMM":
        lr_stat = 2 * (model_int.llf - model_main.llf)
        df_main = len(model_main.fe_params)
        df_int = len(model_int.fe_params)
        df_diff = df_int - df_main
        if df_diff <= 0:
            return
        p_inter = stats.chi2.sf(lr_stat, df_diff)
        print(f"--- {y} ~ {p_z} x {m_z} (LMM, n={len(subset)}) ---")
        print(f"  LRT chi2({df_diff}) = {lr_stat:.2f}, p = {p_inter:.4f}")
        if p_inter < 0.05:
            print("  发现显著交互作用")
            print(model_int.summary().tables[1])
    else:
        anova_res = sm.stats.anova_lm(model_main, model_int)
        if "Pr(>F)" not in anova_res.columns or len(anova_res) <= 1:
            return
        p_inter = anova_res["Pr(>F)"].iloc[1]
        if p_inter >= 0.05:
            return
        print(f"--- {y} ~ {p_z} x {m_z} (OLS, n={len(subset)}) ---")
        print(anova_res)
        print("发现交互作用，完整模型结果:")
        print(model_int.summary().tables[1])

    # ---- Johnson-Neyman 简单斜率图 ----
    if do_jn and p_inter is not None and p_inter < 0.05:
        jn_res = _fit_and_jn(predictor, moderator, outcome, mod_df, controls=controls, verbose=True)
        if jn_res is None:
            return

        p_cn = jn_res["p_cn"]
        m_cn = jn_res["m_cn"]
        y_cn = jn_res["y_cn"]
        m_range = jn_res["m_range"]
        simple_slope = jn_res["simple_slope"]
        ci_lo = jn_res["ci_lo"]
        ci_hi = jn_res["ci_hi"]
        jn_points = jn_res["jn_points"]
        m_raw = jn_res["m_raw"]

        # 绘图
        fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(8, 6),
            gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

        ax_top.plot(m_range, simple_slope, "b-", linewidth=2, label="简单斜率")
        ax_top.fill_between(m_range, ci_lo, ci_hi, alpha=0.15, color="b", label="95% CI")
        ax_top.axhline(y=0, color="gray", linestyle="--", linewidth=1)
        for jp in jn_points:
            ax_top.axvline(x=jp, color="#D4A017", linestyle="--", linewidth=1.2, alpha=0.9)
        ax_top.set_ylabel(f"{p_cn} 的简单斜率", fontsize=9)
        ax_top.legend(fontsize=8)
        sig_prop = ((ci_lo > 0) | (ci_hi < 0)).mean()
        p_str = f"p = {p_inter:.3f}" if p_inter >= 0.001 else "p < 0.001"
        ax_top.text(0.98, 0.03, f"{p_str} | 显著区间 {sig_prop:.0%}",
                    transform=ax_top.transAxes, ha="right", va="bottom",
                    fontsize=8, bbox=dict(boxstyle="round,pad=0.2", facecolor="wheat", alpha=0.7))
        ax_top.set_title(f"{p_cn} × {m_cn} → {y_cn}", fontsize=11, fontweight="bold")

        # 下方: 调节变量分布直方图
        ax_bot.hist(m_raw, bins=40, color="lightgray", edgecolor="white")
        ax_bot.set_xlabel(m_cn, fontsize=10)
        ax_bot.set_ylabel("频数", fontsize=8)
        fig.tight_layout()
        plt.show()

    # ---- 分类调节变量简单斜率图 ----
    if plot_cat_slopes and p_inter is not None and p_inter < 0.05:
        simple_slopes_categorical(predictor, moderator, outcome, data=mod_df, controls=controls)


def simple_slopes_categorical(
    predictor, moderator, outcome,
    data=None,
    controls=("age", "gender"),
    alpha=0.05
):
    if data is None:
        data = mod_df

    p_cn = corr_labels_cn.get(predictor, predictor)
    m_cn = corr_labels_cn.get(moderator, moderator)
    y_cn = corr_labels_cn.get(outcome, outcome)

    needed = [predictor, moderator, outcome] + list(controls)
    missing = [c for c in needed if c not in data.columns]
    if missing:
        print(f"跳过 {p_cn} x {m_cn} -> {y_cn}：缺少字段: {missing}")
        return None, None

    subset = data.dropna(subset=needed).copy()
    if len(subset) < 30:
        print(f"样本量过小 ({len(subset)})，跳过 {p_cn} x {m_cn} -> {y_cn}")
        return None, None

    subset[moderator] = subset[moderator].astype("category")

    x_term = f"{predictor}_z" if f"{predictor}_z" in subset.columns else predictor

    control_terms = []
    for c in controls:
        if c in (predictor, moderator, outcome):
            continue
        if c not in subset.columns:
            continue
        if (isinstance(subset[c].dtype, pd.CategoricalDtype)
                or subset[c].dtype == object
                or pd.api.types.is_bool_dtype(subset[c])):
            control_terms.append(f"C({c})")
        else:
            control_terms.append(c)

    full_formula = f"{outcome} ~ {x_term} * C({moderator})"
    if control_terms:
        full_formula += " + " + " + ".join(control_terms)

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    full_model, mtype = _fit_mixed_or_ols(full_formula, subset, has_groups)

    print(f"--- {y_cn} ~ {p_cn} x {m_cn} 简单斜率 ({mtype}) ---")

    if mtype == "LMM":
        params = full_model.fe_params
        cov = pd.DataFrame(
            full_model.cov_params(),
            index=full_model.fe_params.index,
            columns=full_model.fe_params.index
        )
    else:
        params = full_model.params
        cov_raw = full_model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov = cov_raw.loc[params.index, params.index]
        else:
            cov = pd.DataFrame(cov_raw, index=params.index, columns=params.index)

    levels = list(subset[moderator].cat.categories)
    ref_level = levels[0]

    def _slope_contrast(level):
        contrast = pd.Series(0.0, index=params.index)
        if x_term not in contrast.index:
            return None
        contrast[x_term] = 1.0
        if level != ref_level:
            level_tag = f"[T.{level}]"
            candidates = [n for n in params.index
                          if x_term in n and level_tag in n]
            if not candidates:
                return None
            contrast[candidates[0]] = 1.0
        return contrast

    slope_rows = []
    slope_vectors = {}
    df_resid = getattr(full_model, "df_resid", max(1, len(subset) - len(params)))
    tcrit = stats.t.ppf(1 - alpha / 2, df_resid)

    for level in levels:
        L = _slope_contrast(level)
        if L is None:
            print(f"  无法识别水平 {level} 的简单斜率，跳过")
            continue

        slope = float(L @ params)
        var = float(L @ cov.values @ L.values)
        se = np.sqrt(var) if var >= 0 else np.nan
        tval = slope / se if (se and not np.isnan(se)) else np.nan
        pval = 2 * stats.t.sf(abs(tval), df_resid) if not np.isnan(tval) else np.nan

        ci_lo = slope - tcrit * se if not np.isnan(se) else np.nan
        ci_hi = slope + tcrit * se if not np.isnan(se) else np.nan

        slope_rows.append({
            "level": level, "slope": slope, "se": se,
            "t": tval, "p": pval, "ci_lo": ci_lo, "ci_hi": ci_hi
        })
        slope_vectors[level] = L

    slope_df = pd.DataFrame(slope_rows)
    if len(slope_df):
        slope_df["p_bonf"] = multipletests(slope_df["p"].values, method="bonferroni")[1]
        slope_df["p_fdr"] = multipletests(slope_df["p"].values, method="fdr_bh")[1]

    print(f"--- {y_cn} 在 {m_cn} 各水平下的简单斜率 ---")
    print(slope_df.round(4))

    # Pairwise slope comparisons
    pair_rows = []
    if len(slope_vectors) >= 2:
        for a, b in combinations(levels, 2):
            if a not in slope_vectors or b not in slope_vectors:
                continue
            D = slope_vectors[a] - slope_vectors[b]
            diff = float(D @ params)
            var = float(D @ cov.values @ D.values)
            se = np.sqrt(var) if var >= 0 else np.nan
            tval = diff / se if (se and not np.isnan(se)) else np.nan
            pval = 2 * stats.t.sf(abs(tval), df_resid) if not np.isnan(tval) else np.nan
            ci_lo = diff - tcrit * se if not np.isnan(se) else np.nan
            ci_hi = diff + tcrit * se if not np.isnan(se) else np.nan
            pair_rows.append({
                "level1": a, "level2": b, "slope_diff": diff,
                "se": se, "t": tval, "p": pval,
                "ci_lo": ci_lo, "ci_hi": ci_hi
            })

    pair_df = pd.DataFrame(pair_rows)
    if len(pair_df):
        pair_df["p_bonf"] = multipletests(pair_df["p"].values, method="bonferroni")[1]
        pair_df["p_fdr"] = multipletests(pair_df["p"].values, method="fdr_bh")[1]
        print(f"--- {y_cn} 的简单斜率组间比较 ---")
        print(pair_df.round(4))

    # ---- 交互效应图：每个类别一条回归线 ----
    fig, ax = plt.subplots(figsize=(6, 5))

    x_vals = subset[x_term]
    x_range = np.linspace(x_vals.min(), x_vals.max(), 100)

    if moderator == "cluster":
        level_labels = {lvl: cluster_name_map.get(int(lvl), str(int(lvl))) for lvl in levels}
    else:
        level_labels = {lvl: corr_labels_cn.get(lvl, str(lvl)) for lvl in levels}

    colors = plt.cm.Set2(np.linspace(0, 1, len(levels)))

    for i, level in enumerate(levels):
        # 该水平的截距：参照水平 = Intercept，其他水平 = Intercept + C(moderator)[T.level]
        intercept = params.get("Intercept", 0.0)
        if level != ref_level:
            for key in params.index:
                if f"C({moderator})[T.{level}]" in key and ":" not in key:
                    intercept += params[key]
                    break

        slope_val = slope_df.loc[slope_df["level"] == level, "slope"].values[0]
        y_pred = intercept + slope_val * x_range

        ax.plot(x_range, y_pred, color=colors[i], linewidth=2.5,
                label=level_labels[level])

        # 该水平下的原始散点（低透明度）
        mask = subset[moderator] == level
        ax.scatter(subset.loc[mask, x_term], subset.loc[mask, outcome],
                   color=colors[i], alpha=0.12, s=8, edgecolors="none")

    ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)
    ax.legend(fontsize=9, title=m_cn)
    ax.set_xlabel(p_cn)
    ax.set_ylabel(y_cn)
    ax.set_title(f"{p_cn} x {m_cn} → {y_cn}")
    fig.tight_layout()
    plt.show()

    return slope_df, pair_df

#### 人格特质

In [ ]:
# 个人特质指标数组
personality_traits = ['cse_total', 'ribs_total', 'dat_score']

for trait in personality_traits:
    do_jn = True
    # 原创性
    interaction_analysis('trial_calls', trait, 'originality', do_jn=do_jn)
    interaction_analysis('first_ai_time', trait, 'originality', do_jn=do_jn)
    interaction_analysis('pre_first_call_ideas', trait, 'originality', do_jn=do_jn)
    interaction_analysis('pre_think_time', trait, 'originality', do_jn=do_jn)
    interaction_analysis('perspective_taking', trait, 'originality', do_jn=do_jn)
    interaction_analysis('affected_by_ai', trait, 'originality', do_jn=do_jn)

    # 流畅性
    interaction_analysis('trial_calls', trait, 'fluency', do_jn=do_jn)
    interaction_analysis('first_ai_time', trait, 'fluency', do_jn=do_jn)
    interaction_analysis('pre_first_call_ideas', trait, 'fluency', do_jn=do_jn)
    interaction_analysis('pre_think_time', trait, 'fluency', do_jn=do_jn)
    interaction_analysis('perspective_taking', trait, 'fluency', do_jn=do_jn)
    interaction_analysis('affected_by_ai', trait, 'fluency', do_jn=do_jn)

    # 高质量回答数
    interaction_analysis('trial_calls', trait, 'above_median', do_jn=do_jn)
    interaction_analysis('first_ai_time', trait, 'above_median', do_jn=do_jn)
    interaction_analysis('pre_first_call_ideas', trait, 'above_median', do_jn=do_jn)
    interaction_analysis('pre_think_time', trait, 'above_median', do_jn=do_jn)
    interaction_analysis('perspective_taking', trait, 'above_median', do_jn=do_jn)
    interaction_analysis('affected_by_ai', trait, 'above_median', do_jn=do_jn)

    # 高质量回答比率
    interaction_analysis('trial_calls', trait, 'above_median_ratio', do_jn=do_jn)
    interaction_analysis('first_ai_time', trait, 'above_median_ratio', do_jn=do_jn)
    interaction_analysis('pre_first_call_ideas', trait, 'above_median_ratio', do_jn=do_jn)
    interaction_analysis('pre_think_time', trait, 'above_median_ratio', do_jn=do_jn)
    interaction_analysis('perspective_taking', trait, 'above_median_ratio', do_jn=do_jn)
    interaction_analysis('affected_by_ai', trait, 'above_median_ratio', do_jn=do_jn)


#### cluster

In [ ]:
# 原创性
interaction_analysis('trial_calls', 'cluster', 'originality', plot_cat_slopes=True)
interaction_analysis('first_ai_time', 'cluster', 'originality', plot_cat_slopes=True)
interaction_analysis('pre_first_call_ideas', 'cluster', 'originality', plot_cat_slopes=True)
interaction_analysis('pre_think_time', 'cluster', 'originality', plot_cat_slopes=True)
interaction_analysis('perspective_taking', 'cluster', 'originality', plot_cat_slopes=True)
interaction_analysis('affected_by_ai', 'cluster', 'originality', plot_cat_slopes=True)

# 流畅性
interaction_analysis('trial_calls', 'cluster', 'fluency', plot_cat_slopes=True)
interaction_analysis('first_ai_time', 'cluster', 'fluency', plot_cat_slopes=True)
interaction_analysis('pre_first_call_ideas', 'cluster', 'fluency', plot_cat_slopes=True)
interaction_analysis('pre_think_time', 'cluster', 'fluency', plot_cat_slopes=True)
interaction_analysis('perspective_taking', 'cluster', 'fluency', plot_cat_slopes=True)
interaction_analysis('affected_by_ai', 'cluster', 'fluency', plot_cat_slopes=True)

# 高质量回答数
interaction_analysis('trial_calls', 'cluster', 'above_median', plot_cat_slopes=True)
interaction_analysis('first_ai_time', 'cluster', 'above_median', plot_cat_slopes=True)
interaction_analysis('pre_first_call_ideas', 'cluster', 'above_median', plot_cat_slopes=True)
interaction_analysis('pre_think_time', 'cluster', 'above_median', plot_cat_slopes=True)
interaction_analysis('perspective_taking', 'cluster', 'above_median', plot_cat_slopes=True)
interaction_analysis('affected_by_ai', 'cluster', 'above_median', plot_cat_slopes=True)

# 高质量回答比率
interaction_analysis('trial_calls', 'cluster', 'above_median_ratio', plot_cat_slopes=True)
interaction_analysis('first_ai_time', 'cluster', 'above_median_ratio', plot_cat_slopes=True)
interaction_analysis('pre_first_call_ideas', 'cluster', 'above_median_ratio', plot_cat_slopes=True)
interaction_analysis('pre_think_time', 'cluster', 'above_median_ratio', plot_cat_slopes=True)
interaction_analysis('perspective_taking', 'cluster', 'above_median_ratio', plot_cat_slopes=True)
interaction_analysis('affected_by_ai', 'cluster', 'above_median_ratio', plot_cat_slopes=True)

### cluster汇总图

In [ ]:
# ---- cluster 调节效应汇总图 ----

def _fit_cat_interaction(predictor, outcome, moderator="cluster", data=None,
                         controls=["age", "gender"], alpha=0.05):
    """拟合分类调节模型，返回每个 cluster 水平的简单斜率及 CI。"""
    import patsy as _patsy

    if data is None:
        data = mod_df
    p_z = f"{predictor}_z" if f"{predictor}_z" in data.columns else predictor

    subset = data.dropna(subset=[p_z, moderator, outcome] + controls).copy()
    subset[moderator] = subset[moderator].astype("category")
    levels = list(subset[moderator].cat.categories)

    ctrl_terms = []
    for c in controls:
        if c not in subset.columns:
            continue
        if pd.api.types.is_numeric_dtype(subset[c]):
            ctrl_terms.append(c)
        else:
            ctrl_terms.append(f"C({c})")
    ctrl_str = " + ".join(ctrl_terms)

    rhs = f"{p_z} * C({moderator})"
    if ctrl_str:
        rhs += " + " + ctrl_str
    formula = f"{outcome} ~ {rhs}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    model, mtype = _fit_mixed_or_ols(formula, subset, has_groups)

    # ---- 提取参数与协方差 ----
    if mtype == "LMM":
        params = model.fe_params
        cov_raw = model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov_fe = cov_raw.loc[params.index, params.index]
        else:
            tmp = pd.DataFrame(cov_raw, index=model.model.exog_names,
                               columns=model.model.exog_names)
            cov_fe = tmp.loc[params.index, params.index]

        main_rhs = f"{p_z} + C({moderator})"
        if ctrl_str:
            main_rhs += " + " + ctrl_str
        main_formula = f"{outcome} ~ {main_rhs}"
        try:
            model_main, _ = _fit_mixed_or_ols(main_formula, subset, has_groups)
            lr_stat = 2 * (model.llf - model_main.llf)
            df_diff = len(params) - len(model_main.fe_params)
            p_inter = stats.chi2.sf(lr_stat, df_diff)
        except Exception:
            p_inter = np.nan
    else:
        params = model.params
        cov_raw = model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov_fe = cov_raw.loc[params.index, params.index]
        else:
            cov_fe = pd.DataFrame(cov_raw, index=params.index, columns=params.index)

        main_rhs = f"{p_z} + C({moderator})"
        if ctrl_str:
            main_rhs += " + " + ctrl_str
        main_model = ols(f"{outcome} ~ {main_rhs}", data=subset).fit()
        anova_res = sm.stats.anova_lm(main_model, model)
        p_inter = anova_res["Pr(>F)"].iloc[1]

    # ---- 预测 ----
    x_vals = subset[p_z]
    x_range = np.linspace(x_vals.min(), x_vals.max(), 100)

    ctrl_baseline = {}
    for c in controls:
        if c not in subset.columns:
            continue
        if pd.api.types.is_numeric_dtype(subset[c]):
            ctrl_baseline[c] = subset[c].mean()
        else:
            ctrl_baseline[c] = subset[c].mode().iloc[0]

    colors = plt.cm.Set2(np.linspace(0, 1, len(levels)))
    z_crit = stats.norm.ppf(1 - alpha / 2)
    ref_level = levels[0]

    level_lines = []
    slope_info = []

    for i_lvl, level in enumerate(levels):
        pred_df = pd.DataFrame({
            p_z: x_range,
            moderator: pd.Categorical([level] * len(x_range), categories=levels),
        })
        for c, v in ctrl_baseline.items():
            pred_df[c] = v

        design = _patsy.dmatrix(rhs, pred_df, return_type="dataframe")
        for cn in params.index:
            if cn not in design.columns:
                design[cn] = 0.0
        design = design[params.index]

        cov_v = cov_fe.values
        y_pred = design.values @ params.values
        var_pred = np.sum((design.values @ cov_v) * design.values, axis=1)
        se_pred = np.sqrt(np.maximum(var_pred, 0))
        ci_lo = y_pred - z_crit * se_pred
        ci_hi = y_pred + z_crit * se_pred

        label = cluster_name_map.get(int(level), f"簇 {int(level)}")
        level_lines.append({
            "level": level, "label": label, "color": colors[i_lvl],
            "y_pred": y_pred, "ci_lo": ci_lo, "ci_hi": ci_hi,
        })

        # 简单斜率
        slope_val = params.get(p_z, 0)
        inter_key = f"{p_z}:C({moderator})[T.{level}]"
        if level != ref_level:
            slope_val += params.get(inter_key, 0)

        contrast = pd.Series(0.0, index=params.index)
        contrast[p_z] = 1.0
        if level != ref_level and inter_key in contrast.index:
            contrast[inter_key] = 1.0

        var_slope = float(contrast.values @ cov_fe.values @ contrast.values)
        se_slope = np.sqrt(max(var_slope, 0))
        slope_info.append({
            "slope": slope_val, "se": se_slope,
            "ci_lo": slope_val - z_crit * se_slope,
            "ci_hi": slope_val + z_crit * se_slope,
        })

    return {
        "x_range": x_range, "x_raw": subset[p_z], "y_raw": subset[outcome],
        "level_lines": level_lines, "slope_info": slope_info,
        "p_inter": p_inter, "mtype": mtype,
        "p_cn": corr_labels_cn.get(predictor, predictor),
        "y_cn": corr_labels_cn.get(outcome, outcome),
    }


def plot_cluster_moderation_grid(interactions, data=None, n_cols=4,
                                  controls=["age", "gender"],
                                  figsize_per_cell=(2.8, 2.8)):
    """分类调节变量（cluster）交互效应网格图。"""
    if data is None:
        data = mod_df
    valid = []
    for pred, out in interactions:
        res = _fit_cat_interaction(pred, out, data=data, controls=controls)
        if res is not None:
            valid.append(res)
            print(f"  p_inter = {res['p_inter']:.4f}")
        else:
            print(f"  !! 跳过 {pred} -> {out}")

    n = len(valid)
    if n == 0:
        print("无有效结果")
        return None

    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per_cell[0] * n_cols,
                                      figsize_per_cell[1] * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, res in enumerate(valid):
        ax = axes[i]

        ax.scatter(res["x_raw"], res["y_raw"], color=".85", s=3, alpha=0.35,
                   edgecolors="none")

        for ll in res["level_lines"]:
            ax.plot(res["x_range"], ll["y_pred"], color=ll["color"], linewidth=2,
                    label=ll["label"])
            ax.fill_between(res["x_range"], ll["ci_lo"], ll["ci_hi"],
                            color=ll["color"], alpha=0.08)

        # 斜率标注
        # slope_lines = []
        # for j, ll in enumerate(res["level_lines"]):
        #     b_val = res["slope_info"][j]["slope"]
        #     slope_lines.append(ll["label"] + ": b=" + str(round(b_val, 2)))
        # slope_text = "\n".join(slope_lines)
        # ax.text(0.97, 0.97, slope_text, transform=ax.transAxes,
        #         ha="right", va="top", fontsize=6.5,
        #         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.75))

        ax.set_xlabel(res["p_cn"], fontsize=9)
        ax.set_ylabel(res["y_cn"], fontsize=9)
        ax.legend(fontsize=7, loc="upper left", framealpha=0.8)
        ax.set_title(res["p_cn"] + " x 簇别 → " + res["y_cn"],
                     fontsize=9, fontweight="bold")

        # p 值
        # p_val = res["p_inter"]
        # if np.isnan(p_val):
        #     p_str = "p = N/A"
        # elif p_val >= 0.001:
        #     p_str = "p = %.3f" % p_val
        # else:
        #     p_str = "p < 0.001"
        # ax.text(0.03, 0.97, p_str, transform=ax.transAxes,
        #         ha="left", va="top", fontsize=7.5,
        #         bbox=dict(boxstyle="round,pad=0.2", facecolor="wheat", alpha=0.7))

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.tight_layout()
    return fig


# ===== 流畅性 + 高质量回答数 = 12 项，3×4 网格 =====
all_cluster_interactions = [
    # 流畅性
    ("trial_calls",          "fluency"),
    ("first_ai_time",        "fluency"),
    ("pre_first_call_ideas", "fluency"),
    ("pre_think_time",       "fluency"),
    ("perspective_taking",   "fluency"),
    ("affected_by_ai",       "fluency"),
    # 高质量回答数
    ("trial_calls",          "above_median"),
    ("first_ai_time",        "above_median"),
    ("pre_first_call_ideas", "above_median"),
    ("pre_think_time",       "above_median"),
    ("perspective_taking",   "above_median"),
    ("affected_by_ai",       "above_median"),
    # 原创性
    ("first_ai_time",        "originality"),
]

print("=" * 40)
print("簇别调节效应汇总（流畅性 + 高质量回答数）")
print("=" * 40)
fig_all = plot_cluster_moderation_grid(all_cluster_interactions, mod_df, n_cols=4)
fig_all.savefig("output/moderation_cluster_all.png", dpi=200, bbox_inches="tight")
